# Session 1: Foundations & Prompt Engineering

**Great American Insurance Group — AI Developer Training**

Welcome! Over the next 90 minutes you'll go from making your first LLM call to building a prompt that extracts structured underwriting data from raw text — the same kind of pattern you'll build on in later sessions (retrieval, tools/agents, evals).

Throughout the notebook you'll see:
- 📝 **Exercise** / **Try It Yourself** — write or modify something yourself
- 💬 **Reflection** — a question to think about or discuss
- 💡 **Key Takeaway** — the main point to walk away with

This is not a Python course — the code here is intentionally minimal. The point is the *prompts*, not the plumbing. All LLM calls in this notebook go through one helper function, `get_client().generate(...)`, so you can focus on what you're asking the model rather than how the call is made.

Before starting, make sure you've completed the steps in this folder's `setup.md`.

## Section 1 — Setup & First Model Call

**Goal:** Confirm your environment works, make your first LLM call, and see what a response actually looks like — text, token usage, and model name.

### Step 1.1 — Make your first call

In [1]:
from shared.llm import get_client

client = get_client()

result = client.generate("Say hello in one sentence.")

print("Response:", result.text)
print("Model:", result.model)
print("Tokens - input:", result.usage.input_tokens, "| output:", result.usage.output_tokens, "| total:", result.usage.total_tokens)


Response: Hello there! I hope you're having a wonderful day! 😊
Model: us.anthropic.claude-sonnet-4-6
Tokens - input: 13 | output: 18 | total: 31


💡 **Key Takeaway** — Every LLM call in this training returns the same shape (`text` + `usage` + `model`) regardless of provider. That's the foundation everything else in the lab builds on.

🔧 **Try It Yourself — Exercise 1.1**

Change the message below to something else (ask a question, request a list, whatever you like) and re-run. Notice how the token counts change based on what you asked and how long the response is.

💬 **Reflection** — `input_tokens` and `output_tokens` are billed separately and usually at different rates. Why might a provider charge more for output tokens than input tokens?

In [2]:
# Try changing this message, then re-run the cell
result = client.generate("YOUR MESSAGE HERE")

print("Response:", result.text)
print("Tokens - input:", result.usage.input_tokens, "| output:", result.usage.output_tokens)


Response: It looks like your message came through as **"YOUR MESSAGE HERE"** – which appears to be a placeholder that wasn't filled in. 😊

Could you tell me what you'd actually like to ask or discuss? I'm happy to help!
Tokens - input: 10 | output: 55


---

## Section 2 — Prompt Anatomy: System vs. User

**Goal:** Understand the difference between the **system prompt** (stable instructions about who the model is and how it should behave) and the **user prompt** (the task-specific input, which changes every call).

A well-structured system prompt usually includes: a **role**, a **clear task**, **relevant context**, **constraints/guardrails**, **examples** (if needed), and the **expected output format**.

Below, the same customer message is sent three ways: with no system prompt, a vague one, and a well-structured one. Run all three and compare.

### Variant 1 — No system prompt

In [3]:
customer_message = (
    "My basement flooded last night after the storm. There's about a foot of "
    "standing water and some of it got into the room where we keep the water heater. "
    "What do I do?"
)

no_system = client.generate(customer_message)
print("--- No system prompt ---")
print(no_system.text)


--- No system prompt ---
This is a serious situation - here's what to prioritize:

## Immediate Safety First

**Before entering the basement:**
- **Do NOT enter standing water if electrical panels, outlets, or appliances are submerged or nearby** - electrocution risk is real and deadly
- Turn off electricity to the basement at your main breaker panel if you can do so safely from a dry location
- If you're unsure, call your electric utility company - they can cut power

**For the gas water heater:**
- If you smell gas, leave immediately and call your gas company
- Even without a gas smell, water damage to a gas appliance is serious

## Getting Water Out

- Rent or buy a submersible pump (wet/dry vac for smaller amounts)
- Consider calling a water damage restoration company - they have equipment and can assess damage properly

## After Water is Removed

- Document everything with photos for insurance
- Run dehumidifiers and fans - mold starts growing within 24-48 hours
- **Have the water

### Variant 2 — Vague system prompt

In [4]:
vague_system = "You are a helpful assistant."
vague = client.generate(customer_message, system_prompt=vague_system)
print("--- Vague system prompt ---")
print(vague.text)


--- Vague system prompt ---
This is a stressful situation. Here's what to prioritize:

## Safety First - Do This Before Anything Else

**Do NOT enter the flooded basement if:**
- You're not certain the electricity is off
- The water heater or electrical panel is in the flooded area
- You smell gas

**Immediately:**
- Turn off electricity to the basement at your main breaker panel (if you can reach it safely from a dry area)
- If you smell gas, leave the house and call your gas company
- Don't use light switches or outlets near water

## Once It's Safe to Enter

- Contact your utility companies if you suspect gas or electrical damage
- Document everything with photos/video for insurance
- Call your homeowner's insurance to start a claim
- Start removing water - sump pump, wet/dry vac, or a water removal service

## The Water Heater Specifically

- **Don't relight a gas water heater** that's been flooded until a technician inspects it
- **Don't turn on an electric water heater** that's b

### Variant 3 — Well-structured system prompt

In [5]:
structured_system = (
    "You are a claims intake assistant for a property & casualty insurer. "
    "Your job is to acknowledge the customer's situation, give brief safety guidance "
    "for water damage, and tell them the next step is to file a claim with their adjuster. "
    "Do not give a coverage determination or estimate a payout. "
    "Keep the response under 4 sentences."
)
structured = client.generate(customer_message, system_prompt=structured_system)
print("--- Structured system prompt ---")
print(structured.text)


--- Structured system prompt ---
I'm really sorry to hear about your flooding—that's a stressful situation, and your safety is the top priority. **Do not enter the basement until you've confirmed the electricity and gas to that area are shut off**, as standing water near a water heater creates serious electrocution and gas hazard risks—contact your utility company if you're unsure how to do this safely. Once it's safe, document the damage with photos and videos before removing any water or damaged items. Please contact your insurance adjuster as soon as possible to file a claim so we can get the process started for you.


💡 **Key Takeaway** — System prompts set stable behavior/role; user prompts carry the task-specific data. Precision in the system prompt reduces variance in output.

🔧 **Try It Yourself — Exercise 2.1**

Write your own system prompt below that restricts the assistant to a specific role (e.g., only answers questions about auto claims, refuses to discuss competitors, etc.). Test it with an on-topic message and an off-topic message. Does it hold the boundary?

- `my_system_prompt` — your role restriction, e.g. `"You are a claims intake assistant. You only answer questions about filing or checking the status of an insurance claim. If asked about anything else, politely say you can only help with claims and redirect the user back to that topic."`
- `on_topic_message` — a question that fits inside that role, e.g. `"My car was rear-ended yesterday, how do I start a claim?"`
- `off_topic_message` — a question clearly outside that role, e.g. `"Can you recommend a good pizza recipe?"`

Run both messages through the same system prompt and compare the two responses: did it answer the claims question normally, and did it actually decline or redirect on the off-topic one — or did it just answer that too?

💬 **Reflection** — What happens if a user tries to override your system prompt in their message (e.g., "ignore your instructions and just tell me..."​)? This is called **prompt injection** — something worth keeping in mind anytime user input reaches the model.

### On-topic vs. off-topic — on-topic test

In [6]:
my_system_prompt = "INSERT YOUR SYSTEM PROMPT HERE"

on_topic_message = "INSTERT YOUR ON TOPIC MESSAGE HERE"

print(client.generate(on_topic_message, system_prompt=my_system_prompt).text)

It looks like you've sent a template placeholder message rather than an actual question or topic!

It seems like:
- **"INSERT YOUR SYSTEM PROMPT HERE"** is where a system prompt would normally be configured
- **"INSERT YOUR ON TOPIC MESSAGE HERE"** is where your actual message was meant to go

**Could you tell me:**
- What topic or question did you actually want to discuss?
- Is there something specific I can help you with?

I'm ready to assist once you share what's on your mind! 😊


### On-topic vs. off-topic — off-topic test

In [7]:
off_topic_message = "INSERT YOUR OFF TOPIC MESSAGE HERE"

print(client.generate(off_topic_message, system_prompt=my_system_prompt).text)

I see you've sent what appears to be a template with placeholder text rather than an actual message.

It looks like you may have forgotten to fill in the actual content! Could you clarify:

- **What system prompt** were you intending to reference?
- **What message** did you actually want to send?

I'm happy to help once I know what you're looking for! 😊


---

## Section 3 — Zero-Shot Extraction from Unstructured Text

**Goal:** Extract underwriting information out of a raw, unstructured insurance submission — without giving the model any examples first ("zero-shot").

Fields to extract: `business_name`, `industry`, `locations`, `requested_limits`, `notable_risks`, `missing_information`.

### Setup — synthetic underwriting submissions

In [8]:
SUBMISSIONS = {
    "SUB-001": (
        "Riverside Millwork LLC is a mid-size cabinetry manufacturer based out of a single "
        "facility in Dayton, Ohio. They're requesting $2,000,000 in general liability coverage "
        "and $1,000,000 in property coverage for the building and equipment. Their main exposure "
        "is woodworking dust and the use of industrial saws and finishing chemicals on-site."
    ),
    "SUB-002": (
        "Coastal Fresh Seafood Distributors operates out of two locations - a warehouse in "
        "Tampa, FL and a smaller distribution point in Savannah, GA. They are seeking "
        "$5,000,000 in general liability and want to know about cold storage equipment "
        "breakdown coverage. Notable risk: forklift traffic in the warehouse and ammonia-based "
        "refrigeration systems at both sites."
    ),
    "SUB-003": (
        "Request for BrightPath Tutoring Services, a small business with one office location "
        "in Austin, TX. They tutor K-12 students in person and want $1,000,000 in general "
        "liability and $500,000 in professional liability. No unusual physical risks noted "
        "beyond general foot traffic in the office."
    ),
    "SUB-004": (
        "Summit Ridge Landscaping, based in Denver, CO, is applying for general liability "
        "coverage. They operate commercial mowing equipment, use pesticides/herbicides, and "
        "occasionally do small retaining wall installations. They did not specify a requested "
        "coverage limit in their submission - underwriter should follow up."
    ),
    "SUB-005": (
        "Northgate Auto Body Repair operates a single shop in Columbus, OH. They're requesting "
        "$3,000,000 in general liability and $2,000,000 in garagekeepers coverage. Risks include "
        "vehicle fires during welding/paint work and hazardous waste (used oil, solvents) "
        "storage on-site."
    ),
}

for sub_id, text in SUBMISSIONS.items():
    print(f"{sub_id}: {text[:80]}...")


SUB-001: Riverside Millwork LLC is a mid-size cabinetry manufacturer based out of a singl...
SUB-002: Coastal Fresh Seafood Distributors operates out of two locations - a warehouse i...
SUB-003: Request for BrightPath Tutoring Services, a small business with one office locat...
SUB-004: Summit Ridge Landscaping, based in Denver, CO, is applying for general liability...
SUB-005: Northgate Auto Body Repair operates a single shop in Columbus, OH. They're reque...


🔧 **Try It Yourself — Exercise 3.1**

Write a zero-shot extraction prompt below and run it against `SUB-001` and `SUB-004`. Look closely at what happens on `SUB-004`: does the model correctly say the requested limit is missing, or does it guess a number?

In [9]:
zero_shot_system = (
    "YOUR ZERO-SHOT EXTRACTION SYSTEM PROMPT HERE - describe the fields to extract: "
    "business_name, industry, locations, requested_limits, notable_risks, missing_information"
)

for sub_id in ["SUB-001", "SUB-004"]:
    result = client.generate(SUBMISSIONS[sub_id], system_prompt=zero_shot_system)
    print(f"--- {sub_id} ---")
    print(result.text, "\n")


--- SUB-001 ---
```json
{
  "business_name": "Riverside Millwork LLC",
  "industry": "Cabinetry Manufacturing / Millwork",
  "locations": [
    {
      "description": "Single facility",
      "city": "Dayton",
      "state": "Ohio"
    }
  ],
  "requested_limits": {
    "general_liability": "$2,000,000",
    "property": "$1,000,000",
    "notes": "Property limit intended to cover building and equipment"
  },
  "notable_risks": [
    "Woodworking dust accumulation — fire and respiratory/air quality hazard",
    "Industrial saw operation — employee injury and equipment damage exposure",
    "Finishing chemicals on-site — flammability, storage, and environmental liability concerns",
    "Single-location concentration risk — no redundancy if facility is impaired",
    "Manufacturing operations generally carry elevated workers compensation and products liability exposure"
  ],
  "missing_information": [
    "Annual revenue and payroll figures",
    "Number of employees",
    "Details on dus

💡 **Key Takeaway** — Zero-shot extraction is a reasonable starting point but is inconsistent on formatting and prone to inventing values for missing fields unless explicitly instructed otherwise.

---

## Section 4 — Few-Shot Prompting

**Goal:** See whether giving the model worked examples ("few-shot") improves consistency enough to justify the extra tokens those examples cost.

🔧 **Try It Yourself — Exercise 4.1**

Add 1–2 worked examples (an example submission text, and the correctly extracted fields for it — including one example showing how a missing field should be handled) to your extraction prompt below. Re-run against `SUB-001` and `SUB-004` and compare against Section 3's output. Also compare the `input_tokens` between the two versions — how much more did the examples cost?

In [10]:
few_shot_system = (
    zero_shot_system + "\n\n"
    "Here are two examples:\n\n"
    "Submission: BrightPath Tutoring Services, one office in Austin, TX. They tutor "
    "K-12 students in person and want $1,000,000 general liability and $500,000 "
    "professional liability. No unusual risks.\n"
    "Output: business_name: BrightPath Tutoring Services, industry: Tutoring, "
    "locations: Austin TX, requested_limits: $1,000,000 GL / $500,000 professional "
    "liability, notable_risks: none noted, missing_information: none\n\n"
    "Submission: Green Valley Pet Grooming, single location in Boise, ID. They groom "
    "dogs and cats. No coverage amount was mentioned.\n"
    "Output: business_name: Green Valley Pet Grooming, industry: Pet Grooming, "
    "locations: Boise ID, requested_limits: not specified, notable_risks: none noted, "
    "missing_information: requested coverage limit was not given"
)

for sub_id in ["SUB-001", "SUB-004"]:
    result = client.generate(SUBMISSIONS[sub_id], system_prompt=few_shot_system)
    print(f"--- {sub_id} (few-shot) ---")
    print(result.text)
    print("input_tokens:", result.usage.input_tokens, "\n")


--- SUB-001 (few-shot) ---
business_name: Riverside Millwork LLC, industry: Cabinetry Manufacturing, locations: Dayton OH, requested_limits: $2,000,000 GL / $1,000,000 property, notable_risks: woodworking dust accumulation, industrial saw operation, use of finishing chemicals on-site, missing_information: none
input_tokens: 334 

--- SUB-004 (few-shot) ---
business_name: Summit Ridge Landscaping, industry: Landscaping, locations: Denver CO, requested_limits: not specified, notable_risks: use of pesticides/herbicides (environmental/chemical liability exposure); operation of commercial mowing equipment (bodily injury/property damage exposure); retaining wall installations (structural liability exposure), missing_information: requested coverage limit was not provided – underwriter follow-up required
input_tokens: 312 



💡 **Key Takeaway** — Examples materially improve consistency and formatting, but each example adds input tokens — few-shot is a deliberate cost/quality trade-off, not a free upgrade.

---

## Section 5 — Structured Outputs

**Goal:** So far the model has been returning free text. For downstream application code to actually use this, we need predictable data with an explicit schema — including how missing values should be represented (e.g., `null`). Modern LLM APIs enforce this through native structured-output mechanisms, eliminating manual parsing.

### Guided demo — native structured output with Pydantic

The `SubmissionExtraction` class below uses Pydantic to define the exact schema we want the model to return. The API will validate that the model's output matches this schema — no manual JSON parsing needed.

In [11]:
from pydantic import BaseModel

class SubmissionExtraction(BaseModel):
    """Extraction schema for underwriting submissions.

    The API guarantees that every field returned matches this exact structure.
    """
    business_name: str
    industry: str
    locations: list[str]
    requested_limits: str | None = None
    notable_risks: list[str]
    missing_information: list[str]

system_prompt = (
    "Extract underwriting information from the submission text. "
    "For the 'requested_limits' field, if no specific amount is mentioned, set it to null. "
    "For 'notable_risks' and 'missing_information', use empty lists if there are none. "
    "Do not invent values for missing fields."
)

extracted = {}
for sub_id, text in SUBMISSIONS.items():
    result = client.generate_structured(text, SubmissionExtraction, system_prompt=system_prompt)
    extracted[sub_id] = result

for sub_id, data in extracted.items():
    print(f"--- {sub_id} ---")
    print(data.model_dump_json(indent=2))
    print()

--- SUB-001 ---
{
  "business_name": "Riverside Millwork LLC",
  "industry": "Cabinetry Manufacturing / Millwork",
  "locations": [
    "Dayton, Ohio"
  ],
  "requested_limits": "$2,000,000 General Liability; $1,000,000 Property (Building and Equipment)",
  "notable_risks": [
    "Woodworking dust accumulation and combustion risk",
    "Use of industrial saws presenting machinery and operator injury hazards",
    "On-site use of finishing chemicals creating fire, explosion, and toxic exposure risks"
  ],
  "missing_information": [
    "Annual revenue or payroll figures",
    "Number of employees",
    "Details on dust collection and ventilation systems",
    "Chemical storage and handling procedures",
    "Prior loss history",
    "Building age and construction type",
    "Value of equipment/machinery for property coverage accuracy"
  ]
}

--- SUB-002 ---
{
  "business_name": "Coastal Fresh Seafood Distributors",
  "industry": "Seafood Distribution",
  "locations": [
    "Tampa, FL (wa

🔧 **Try It Yourself — Exercise 5.1**

The schema is defined above in the `SubmissionExtraction` class. If the extraction for any submission seems incomplete or inaccurate, modify the `system_prompt` above to clarify the instructions (e.g., better guidance on what counts as a "risk") and re-run. You can also edit the field descriptions in the `SubmissionExtraction` class to provide additional schema-level guidance to the model.

💬 **Reflection** — How does specifying the schema upfront (before asking the model) compare to asking for JSON and parsing it yourself?

💡 **Key Takeaway** — Native structured-output APIs enforce schema conformance for you — the API guarantees the shape is correct. Your job is to define the schema clearly and handle the semantics (validators check that extracted values make sense, not just that they fit the shape).

### 5.2 — Adding Validators

Schema conformance (the right shape) is different from semantic correctness (the values make sense). Pydantic validators let you enforce business rules on extracted data — catching bad extractions even when the shape is already valid.

Below, we extend `SubmissionExtraction` with a validator that checks each location matches a `"City, ST"` format.

In [13]:
import re
from pydantic import field_validator, ValidationError

class SubmissionExtractionWithValidators(BaseModel):
    """Extended extraction schema with semantic validators."""
    business_name: str
    industry: str
    locations: list[str]
    requested_limits: str | None = None
    notable_risks: list[str]
    missing_information: list[str]

    @field_validator('locations', mode='before')
    @classmethod
    def validate_locations(cls, v):
        """Each location should follow 'City, ST' format."""
        if not isinstance(v, list):
            v = [v]
        for loc in v:
            if not re.match(r'^[A-Za-z\s]+,\s*[A-Z]{2}$', loc):
                raise ValueError(f"Location '{loc}' does not match 'City, ST' format")
        return v

# Re-run extraction with the validator-enabled schema
print("Running extraction with validators enabled...\n")
extracted_validated = {}
for sub_id, text in SUBMISSIONS.items():
    try:
        result = client.generate_structured(text, SubmissionExtractionWithValidators, system_prompt=system_prompt)
        extracted_validated[sub_id] = result
        print(f"✓ {sub_id}: passed validation")
    except ValidationError as e:
        print(f"✗ {sub_id}: VALIDATION ERROR")
        print(f"  {e}")

print("\nValid extractions:")
for sub_id, data in extracted_validated.items():
    print(f"--- {sub_id} ---")
    print(f"  locations: {data.locations}")
    print()

Running extraction with validators enabled...



RuntimeError: Bedrock returned tool input that failed schema validation: 1 validation error for SubmissionExtractionWithValidators
locations
  Value error, Location 'Dayton, Ohio' does not match 'City, ST' format [type=value_error, input_value=['Dayton, Ohio'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

🔧 **Try It Yourself — Exercise 5.2**

Edit the regex in the `validate_locations` validator above to be more or less strict about location formats. For instance, you could allow locations with full state names instead of abbreviations, or allow additional punctuation. Try variations and re-run — see which submissions pass or fail.

💬 **Reflection** — What happens when the model's extraction fails validation? Should you retry the call, adjust the prompt, or relax the validator?

💡 **Key Takeaway** — Validators are your second line of defense. The API guarantees schema conformance; validators guarantee semantic correctness. Combine clear prompts + schema-level guidance with field-level validation rules to catch and handle extraction problems at runtime.

---

## Section 6 — Token Cost & Prompting Best Practices

**Goal:** Longer prompts and more examples aren't automatically better. Below is a deliberately bloated version of the JSON extraction prompt — redundant instructions, restated rules, unnecessary padding.

### Variant 1 — Verbose prompt

In [ ]:
verbose_system = (
    "You are an extremely helpful and knowledgeable AI assistant with expertise in "
    "the insurance underwriting domain, and you have been carefully trained to assist "
    "with a wide variety of tasks related to reading and understanding insurance submissions. "
    "When you receive an insurance submission, it is very important that you read it "
    "carefully and thoroughly before responding. Your task, which is very important, is to "
    "extract the underwriting information from the submission. Please make sure to extract "
    "the business name, and also the industry, and also the locations, and also the requested "
    "limits, and also the notable risks, and also note down any information that seems to be "
    "missing from the submission. It is critical and very important that you extract the "
    "data according to the schema provided. Remember, if a field is not present in the text, "
    "use null for optional string fields or empty lists for array fields, and please remember "
    "to add a note about it to missing_information. Please do not invent or make up any values "
    "that are not actually present in the submission text, this is very important. Thank you "
    "for your help with this important task."
)

verbose_result = client.generate_structured(SUBMISSIONS["SUB-002"], SubmissionExtraction, system_prompt=verbose_system)
print("Verbose prompt - input_tokens:", verbose_result.model_dump_json().count("\""))
print(verbose_result.model_dump_json(indent=2))

### Variant 2 — Concise prompt (from Section 5)

In [ ]:
concise_result = client.generate_structured(SUBMISSIONS["SUB-002"], SubmissionExtraction, system_prompt=system_prompt)
print("Concise prompt - input_tokens:", concise_result.model_dump_json().count("\""))
print(concise_result.model_dump_json(indent=2))

🔧 **Try It Yourself — Exercise 6.1**

Shorten `verbose_system` above while keeping output quality on `SUB-002`. Compare `input_tokens` and output quality against the original `json_system` from Section 5.

💡 **Key Takeaway — Best practices to carry forward:**
- Define the task clearly
- Provide only relevant context
- Separate instructions from source data
- Specify the expected output format
- State how missing information should be handled
- Prevent unsupported assumptions
- Remove duplicated or conflicting instructions
- Request only the output the application actually needs
- Use few-shot examples only when they materially improve results

---

## Section 7 — Final Challenge

Here's a new submission you haven't seen yet. Build a prompt from scratch that:
- Uses system and user messages appropriately
- Extracts the required fields
- Returns valid JSON
- Identifies missing information (there is at least one gap in this submission)
- Avoids unsupported assumptions
- Minimizes unnecessary tokens
- Follows the best practices above

Tip: you don't have to start from a blank page — reusing and adapting your `json_system` from Section 5 is a reasonable starting point.

🔧 **Try It Yourself — Exercise 7.1**

Write your final prompt below and run it against `SUB-006`.

In [ ]:
SUB_006 = (
    "Lakeside Family Dentistry is requesting professional liability coverage for their "
    "single-location practice in Madison, WI. They have 4 dentists and 6 support staff. "
    "Notable risk: use of nitrous oxide sedation and on-site X-ray equipment."
)

final_system = "YOUR FINAL SYSTEM PROMPT HERE"

result = client.generate(SUB_006, system_prompt=final_system)
print(result.text)
print("input_tokens:", result.usage.input_tokens)


💡 **Key Takeaway** — A working checklist for evaluating any extraction prompt going forward, before writing one from scratch: system/user separation, explicit schema, explicit missing-value handling, no unsupported assumptions, token discipline.

---

## Wrap-Up

💬 **Reflection** — Looking back at everything above, what would you monitor if this extraction prompt were running in production? Some things worth considering: how often required fields come back missing/null, how token usage grows as submissions get longer, and how consistent the output is across similar submissions.

**Next session:** we'll take this same extraction pattern and connect it to a document store — building a chat app that can answer underwriting questions over real ACORD-style submission documents using retrieval (RAG).